# 第34章 子图与组合图（subplots）

使用subplots和GridSpec把多个相关图组织为共享阅读结构。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。学习重点不是“把图画出来”，而是让图表服务于一个可回答的问题。


## 适用场景

同一分析需要多个互补图表，或需要比较小倍图。

## 数据结构

多个共享维度或相关指标的数据集。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 sharex=True 改为 sharex=False，观察独立坐标轴与共享坐标轴的差异
2. 修改 gridspec_kw 中的 width_ratios 为 [1, 1]，对比均等与非均等列宽布局
3. 调整 figsize 参数（如 (12, 5) 或 (10, 6)），说明画布尺寸对子图可读性的影响


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from js import window
base_url = window.location.origin
transactions = pd.read_csv(f"{base_url}/datasets/uci_online_retail_200k.csv", parse_dates=["InvoiceDate"])
transactions["amount"] = transactions["Quantity"] * transactions["UnitPrice"]
transactions["month"] = transactions["InvoiceDate"].dt.to_period("M").astype("string")
completed = transactions.query("Quantity > 0 and UnitPrice > 0")
monthly_summary = completed.groupby("month").agg(sales=("amount", "sum"), orders=("InvoiceNo", "nunique"))
months = monthly_summary.index.to_numpy()
sales = (monthly_summary["sales"] / 10_000).to_numpy()
orders = monthly_summary["orders"].to_numpy()
profit = sales * 0.18
top_countries = completed.groupby("Country")["amount"].sum().nlargest(4).index
country_rows = transactions[transactions["Country"].isin(top_countries)].copy()
country_rows["flow"] = np.where(country_rows["Quantity"] > 0, "销售", "退货")
country_rows["amount_abs"] = country_rows["amount"].abs()
regional_summary = country_rows.pivot_table(index="Country", columns="flow", values="amount_abs", aggfunc="sum", fill_value=0) / 10_000
regions = regional_summary.index.to_numpy()
online = regional_summary.get("销售", pd.Series(0, index=regional_summary.index)).to_numpy()
offline = regional_summary.get("退货", pd.Series(0, index=regional_summary.index)).to_numpy()
samples = completed["amount"].sample(2_000, random_state=25).to_numpy()
print(f"UCI Online Retail：{len(transactions):,} 行；图表使用聚合结果与固定样本")


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8.5, 6), sharex=True)
axes[0].plot(months, sales, marker="o", color="#1a73e8")
axes[0].set(title="销售额", ylabel="万元")
axes[1].plot(months, profit, marker="s", color="#188038")
axes[1].set(title="利润", xlabel="月份", ylabel="万元")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.18)
fig.suptitle("上半年经营指标", fontsize=16)
fig.tight_layout()
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
fig = plt.figure(figsize=(10, 5), layout="constrained")
grid = fig.add_gridspec(2, 2, width_ratios=[2, 1])
ax_trend = fig.add_subplot(grid[:, 0])
ax_region = fig.add_subplot(grid[0, 1])
ax_channel = fig.add_subplot(grid[1, 1])
ax_trend.plot(months, sales, marker="o", color="#1a73e8")
ax_trend.set(title="月度销售趋势", ylabel="万元")
ax_region.barh(regions, online + offline, color="#188038")
ax_region.set(title="区域总量")
ax_channel.pie([online.sum(), offline.sum()], labels=["线上", "线下"], autopct="%.0f%%", colors=["#1a73e8", "#f9ab00"])
ax_channel.set_title("渠道构成")
plt.show()


## 3. 参数说明

- nrows/ncols：网格
- sharex/sharey：共享轴
- gridspec_kw：比例
- suptitle：画布标题


## 4. 结果解读

先阅读总标题，再按固定顺序逐图比较；共享轴时确认量纲一致。


## 常见误区

- 每个子图重复图例和标签
- 子图尺寸太小
- 双Y轴制造虚假同步


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
axes[0].plot(months, orders, marker="o", color="#1a73e8")
axes[0].set(title="订单趋势")
axes[1].bar(regions, online, color="#188038")
axes[1].set(title="线上销售")
axes[2].hist(samples, bins=14, color="#f9ab00", edgecolor="white")
axes[2].set(title="订单金额分布")
fig.suptitle("经营分析面板")
fig.tight_layout()
plt.show()


## 本章小结

使用subplots和GridSpec把多个相关图组织为共享阅读结构。


### 你已经掌握

- 判断子图与组合图（subplots）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 同一分析需要多个互补图表，或需要比较小倍图。 |
| 数据结构 | 多个共享维度或相关指标的数据集。 |
| 结果解读 | 先阅读总标题，再按固定顺序逐图比较；共享轴时确认量纲一致。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `nrows/ncols` | 网格 |
| `sharex/sharey` | 共享轴 |
| `gridspec_kw` | 比例 |
| `suptitle` | 画布标题 |


### 需要注意

- 每个子图重复图例和标签
- 子图尺寸太小
- 双Y轴制造虚假同步


### 完成检查

- [ ] 能判断什么问题适合使用子图与组合图（subplots）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
